In [10]:
import os, shutil
from ase.db import connect
from ase.build import add_adsorbate
from ase.io import read, write

db = connect('final_optimized_db.db')
# 建议输出到新的数据库，避免覆盖原“右下角整体”结果
ads_db = connect('Surfaces_LiS+Li.db')

# 原始平面坐标（Å）与高度/元素
local_xy = [(5,5), (6,6), (4,4)]
z_list   = [4.2,   3.24,  3.24]
symbols  = ['S',   'Li',  'Li'  ]

# 选择要“分解出去”的 Li（索引基于上面三个原子）
li_indices = [i for i, s in enumerate(symbols) if s == 'Li']
move_index = li_indices[0]   # 默认分解第一个 Li；要换另一个就改成 li_indices[1]

# 其余原子（S + 留下的那个 Li）继续作为一个小团簇
stay_indices = [i for i in range(len(symbols)) if i != move_index]

# 用“留下的两个原子”的局部坐标计算其几何中心 → 得到相对偏移
mx = sum(local_xy[i][0] for i in stay_indices) / len(stay_indices)
my = sum(local_xy[i][1] for i in stay_indices) / len(stay_indices)
offset_xy = {i: (local_xy[i][0] - mx, local_xy[i][1] - my) for i in stay_indices}

def frac_to_xy(atoms, fx, fy):
    # 分数坐标 -> 绝对 x–y（适用于非正交晶胞）
    a, b, _ = atoms.get_cell()
    r = fx * a + fy * b
    return float(r[0]), float(r[1])

margin = 0.20  # 与原来一致：边界留 10% 缓冲
for row in db.select():
    slab = row.toatoms().copy()

    # 右下角：团簇（S + 留下的 Li）
    brx, bry = frac_to_xy(slab, 1.0 - margin, margin)
    # 左上角：被分解出去的 Li
    tlx, tly = frac_to_xy(slab, margin, 1.0 - margin)

    # 放置团簇（保持相对几何）
    for i in stay_indices:
        dx, dy = offset_xy[i]
        add_adsorbate(slab, symbols[i], height=z_list[i], position=(brx + dx, bry + dy))

    # 把选中的 Li 单独放到左上角
    add_adsorbate(slab, symbols[move_index], height=z_list[move_index], position=(tlx, tly))

    # 防止越界缠绕到相邻晶胞
    slab.wrap(pbc=[True, True, False])

    ads_db.write(
        slab,
        model=row.formula,
        note='Li split to top-left',
        moved_Li_index=int(move_index),
        margin=margin
    )
